<a href="https://colab.research.google.com/github/Kevin-March/Tesis/blob/testing/evaluacion_ragas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB2 · Evaluación RAGAS (Fase 1)

Evalúa las 171 respuestas generadas por NB1 (`respuestas_gpt4o.json`).

**Entorno AISLADO de NB1**: RAGAS 0.4 exige langchain-core 0.3.x, incompatible con el 1.x de NB1.

**Métricas**: Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecisionWithReference + la custom `deteccion_de_vigencia` (el corazón de la tesis).

**Orden**: correr celda por celda. La celda 5 tiene `MODO_PRUEBA=True` (6 respuestas). Verificar que sale bien y recién ahí `MODO_PRUEBA=False` para las 171.

## 1. Dependencias (pines exactos — no cambiar)

In [1]:
# ============================================================================
# NB2 · evaluacion_ragas.ipynb — CELDA 1: dependencias (entorno AISLADO)
# ============================================================================
# ⚠️ ESTE NOTEBOOK CORRE EN UN ENTORNO SEPARADO DE NB1.
#    RAGAS 0.4.3 exige langchain-core 0.3.x, que es INCOMPATIBLE con el
#    langchain-core 1.x que usa NB1 (LangGraph). Por eso son notebooks distintos:
#    NB1 genera los JSON, NB2 los evalúa. No se ejecutan juntos.
#
# ⚠️ pip install ragas "a secas" está ROTO (bug upstream: importa un módulo de
#    langchain-community que la 0.4 eliminó). Hay que PINEAR estas versiones exactas.
# ----------------------------------------------------------------------------
!pip install -q \
  "ragas==0.4.3" \
  "langchain-core==0.3.86" \
  "langchain==0.3.27" \
  "langchain-community==0.3.27" \
  "langchain-openai==0.3.35" \
  "openai>=1.0.0"

# verificación de que importa (si esto falla, revisar los pines de arriba)
import ragas
print("ragas", ragas.__version__, "— import OK")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take

## 2. Setup: Drive, API key, cargar los JSON de NB1

In [3]:
# ============================================================================
# CELDA 2: setup — Drive, API key, cargar los JSON de NB1
# ============================================================================
import os, json, getpass
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# API key de OpenAI (el juez de RAGAS es GPT-4o)
# Lee la key desde Colab Secrets (ícono de llave 🔑 en la barra izquierda).
# El secret debe llamarse OPENAI_API_KEY y tener "Acceso al notebook" activado.
import os
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("API key cargada desde Colab Secrets ✓")
    except Exception as e:
        # fallback: pedirla a mano si el secret no está o no tiene acceso
        import getpass
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
        print("API key ingresada a mano")

# --- rutas (misma estructura que NB1) ---
CARPETA = "/content/drive/MyDrive/tesis_chatbot/ragas"
PATH_RESP = os.path.join(CARPETA, "respuestas_gpt4o.json")

with open(PATH_RESP, encoding="utf-8") as f:
    data = json.load(f)

META = data["meta"]
RESP = data["respuestas"]
print("Cargado:", PATH_RESP)
print("Meta:", META)
print("Total respuestas:", len(RESP), "(esperado 171 = 57×3)")

# índice rápido por (id, brazo)
POR_CLAVE = {(r["id"], r["brazo"]): r for r in RESP}
BRAZOS = sorted({r["brazo"] for r in RESP})
IDS = sorted({r["id"] for r in RESP})
print("Brazos:", BRAZOS)
print("Preguntas:", len(IDS))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
API key cargada desde Colab Secrets ✓
Cargado: /content/drive/MyDrive/tesis_chatbot/ragas/respuestas_gpt4o.json
Meta: {'generado': '2026-07-18T18:13:28.285990', 'modo': 'completo', 'n_preguntas': 57, 'n_brazos': 3, 'top_k': 5, 'llm': 'gpt-4o'}
Total respuestas: 171 (esperado 171 = 57×3)
Brazos: ['A_baseline', 'B_graphrag', 'C_agente']
Preguntas: 57


## 3. Juez (GPT-4o) y las 4 métricas estándar

In [4]:
# ============================================================================
# CELDA 3: instanciar el JUEZ (LLM + embeddings) y las MÉTRICAS de RAGAS
# ============================================================================
# El juez es GPT-4o (mismo modelo que generó, pero como los 3 brazos usan el
# mismo generador, el sesgo de auto-preferencia es SIMÉTRICO y se cancela en la
# comparación entre brazos. Documentarlo en Limitaciones).
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasEmbeddings

_client = AsyncOpenAI()   # ASYNC: las métricas usan .ascore() (asíncrono)

# LLM juez y embeddings (para AnswerRelevancy)
judge_llm = llm_factory("gpt-4o", client=_client, max_tokens=4000)  # max_tokens alto: faithfulness descompone respuestas largas
judge_emb = RagasEmbeddings(client=_client, model="text-embedding-3-small")

# --- Las 4 métricas estándar (API collections, RAGAS 0.4) ---
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextRecall, ContextPrecisionWithReference
)
m_faith   = Faithfulness(llm=judge_llm)
m_relev   = AnswerRelevancy(llm=judge_llm, embeddings=judge_emb)
m_recall  = ContextRecall(llm=judge_llm)                 # necesita reference (gold)
m_precis  = ContextPrecisionWithReference(llm=judge_llm)  # necesita reference (gold)

print("Juez y métricas estándar listos.")
print("Firmas (recordatorio):")
print("  Faithfulness.ascore(user_input, response, retrieved_contexts)   -> sin gold")
print("  AnswerRelevancy.ascore(user_input, response)                    -> sin gold")
print("  ContextRecall.ascore(user_input, retrieved_contexts, reference) -> CON gold")
print("  ContextPrecisionWithReference.ascore(user_input, reference, retrieved_contexts) -> CON gold")


Juez y métricas estándar listos.
Firmas (recordatorio):
  Faithfulness.ascore(user_input, response, retrieved_contexts)   -> sin gold
  AnswerRelevancy.ascore(user_input, response)                    -> sin gold
  ContextRecall.ascore(user_input, retrieved_contexts, reference) -> CON gold
  ContextPrecisionWithReference.ascore(user_input, reference, retrieved_contexts) -> CON gold


## 4. Métrica custom: detección de vigencia

La única que detecta el fallo central del baseline (citar ley derogada como vigente). Usa el campo `rubrica` del set gold como criterio.

In [5]:
# ============================================================================
# CELDA 4: MÉTRICA CUSTOM · deteccion_de_vigencia  (el corazón de la tesis)
# ============================================================================
# Ninguna de las 4 métricas estándar detecta el fallo central del baseline:
# citar una ley DEROGADA como si estuviera vigente (Faithfulness incluso premia
# eso si el baseline resume fielmente un contexto viejo). Esta métrica end-to-end
# pregunta directamente: ¿la respuesta enuncia bien el estado de vigencia?
#
# ⚠️ AspectCritic fue ELIMINADO en RAGAS 0.4. Se usa el decorador @discrete_metric.
# La rúbrica de cada pregunta (campo 'rubrica' del set gold) es el CRITERIO que
# se le pasa al juez.
from ragas.metrics import discrete_metric
from ragas.metrics.result import MetricResult
from openai import OpenAI

_c = OpenAI()

PROMPT_VIGENCIA = """Sos un evaluador jurídico. Determiná si la RESPUESTA enuncia correctamente
el estado de vigencia de las normas involucradas, según el CRITERIO dado.

CRITERIO (qué debe cumplir una respuesta correcta):
{rubrica}

HECHOS DE REFERENCIA (verdad sobre la vigencia):
{gold}

RESPUESTA A EVALUAR:
{response}

Respondé con UNA sola palabra:
- "correcto"   si la respuesta respeta el criterio y no presenta como vigente una norma derogada.
- "incorrecto" si presenta una norma derogada como vigente, o contradice el criterio de vigencia.
- "no_aplica"  si la pregunta no involucra vigencia de normas.
Palabra:"""

@discrete_metric(name="deteccion_de_vigencia",
                 allowed_values=["correcto", "incorrecto", "no_aplica"])
def deteccion_de_vigencia(response: str, gold: str, rubrica: str) -> MetricResult:
    # sin criterio ni gold no se puede juzgar vigencia
    if not (rubrica or gold):
        return MetricResult(value="no_aplica", reason="sin rúbrica ni gold de vigencia")
    prompt = PROMPT_VIGENCIA.format(rubrica=rubrica or "(no especificado)",
                                    gold=gold or "(no especificado)",
                                    response=response)
    out = _c.chat.completions.create(
        model="gpt-4o", temperature=0,
        messages=[{"role": "user", "content": prompt}],
    ).choices[0].message.content.strip().lower()
    # OJO: chequear "incorrecto" PRIMERO — contiene "correcto" como substring
    if "incorrecto" in out:
        val = "incorrecto"
    elif "correcto" in out:
        val = "correcto"
    else:
        val = "no_aplica"
    return MetricResult(value=val, reason=out[:120])

# prueba rápida en una respuesta del baseline (debería dar 'incorrecto' en una de reforma)
_demo = POR_CLAVE.get(("P1-01", "A_baseline"))
if _demo:
    r = deteccion_de_vigencia.score(response=_demo["respuesta"],
                                    gold=_demo["gold"], rubrica=_demo["rubrica"])
    print("Demo P1-01 / A_baseline →", r.value, "|", r.reason[:80])


Demo P1-01 / A_baseline → incorrecto | incorrecto


## 5. Ejecutar la evaluación (con guardado incremental)

Guarda tras **cada** respuesta. Si Colab se desconecta (pantalla apagada, timeout), volvé a correr esta misma celda: **retoma donde quedó**, no repite lo hecho.

⚠️ Empezá con `MODO_PRUEBA=True` (6). Si sale bien, `MODO_PRUEBA=False` (171). La corrida completa puede tardar 30-60 min.

In [6]:
# ============================================================================
# CELDA 5: EJECUTAR la evaluación — CON GUARDADO INCREMENTAL Y RESUME
# ============================================================================
# Guarda después de CADA respuesta. Si el runtime se desconecta (pantalla
# apagada, timeout de Colab), volvés a correr esta MISMA celda y RETOMA desde
# donde quedó — no repite lo ya hecho ni re-paga las llamadas.
#
# Reglas especiales (§16.5): fuera_alcance se EXCLUYE de AnswerRelevancy
# (castiga fallback honesto) → flag fallback_ok; gold vacío salta recall/precision.
import asyncio, time, json, os
from datetime import datetime

MODO_PRUEBA = False     # <-- False = las 171. True = 6 de prueba.
N_PRUEBA = 6
PAUSA = 1.5

sufijo = "_PRUEBA" if MODO_PRUEBA else ""
OUT_PATH = os.path.join(CARPETA, f"metricas_ragas{sufijo}.json")

# --- RESUME: cargar lo ya hecho (si existe) y saltearlo ---
resultados = []
hechas = set()
if os.path.exists(OUT_PATH):
    try:
        prev = json.load(open(OUT_PATH, encoding="utf-8"))
        resultados = prev.get("resultados", [])
        hechas = {(r["id"], r["brazo"]) for r in resultados}
        print(f"↻ RESUME: {len(hechas)} respuestas ya evaluadas se saltean.")
    except Exception as e:
        print("No se pudo leer parcial, empiezo de cero:", e)

def es_fuera_alcance(r):  return "fuera_alcance" in (r.get("tipo") or [])
def texto_pregunta(r):    return r.get("pregunta_usada") or r.get("pregunta_original")

def guardar():
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump({"meta": {"generado": datetime.now().isoformat(),
                            "n": len(resultados), "completo": None},
                   "resultados": resultados}, f, ensure_ascii=False, indent=2)

async def evaluar_una(r):
    q, resp, ctxs = texto_pregunta(r), r["respuesta"], r["contextos"]
    gold, rub = r.get("gold") or "", r.get("rubrica") or ""
    fila = {"id": r["id"], "brazo": r["brazo"],
            "afectada_por_reforma": r.get("afectada_por_reforma", False),
            "tipo": r.get("tipo") or []}
    try: fila["faithfulness"] = (await m_faith.ascore(user_input=q, response=resp, retrieved_contexts=ctxs)).value
    except Exception as e: fila["faithfulness"]=None; fila["err_faith"]=str(e)[:80]
    if es_fuera_alcance(r):
        fila["answer_relevancy"]=None
        fila["fallback_ok"]=any(k in resp.lower() for k in
            ["no tengo","no cuento","no dispongo","no puedo","fuera de","no encontré","no hay información"])
    else:
        try: fila["answer_relevancy"]=(await m_relev.ascore(user_input=q, response=resp)).value
        except Exception as e: fila["answer_relevancy"]=None; fila["err_relev"]=str(e)[:80]
    if gold:
        try: fila["context_recall"]=(await m_recall.ascore(user_input=q, retrieved_contexts=ctxs, reference=gold)).value
        except Exception as e: fila["context_recall"]=None; fila["err_recall"]=str(e)[:80]
        try: fila["context_precision"]=(await m_precis.ascore(user_input=q, reference=gold, retrieved_contexts=ctxs)).value
        except Exception as e: fila["context_precision"]=None; fila["err_precis"]=str(e)[:80]
    else:
        fila["context_recall"]=None; fila["context_precision"]=None
    try: fila["deteccion_vigencia"]=deteccion_de_vigencia.score(response=resp, gold=gold, rubrica=rub).value
    except Exception as e: fila["deteccion_vigencia"]=None; fila["err_vig"]=str(e)[:80]
    return fila

async def correr(lista):
    for i, r in enumerate(lista, 1):
        if (r["id"], r["brazo"]) in hechas:
            continue   # ya evaluada en una corrida anterior
        for intento in range(4):
            try:
                fila = await evaluar_una(r)
                resultados.append(fila)
                hechas.add((r["id"], r["brazo"]))
                guardar()               # <-- GUARDADO INCREMENTAL tras cada una
                break
            except Exception as e:
                if "rate" in str(e).lower() and intento < 3:
                    espera = 2**intento; print(f"  429, esperando {espera}s..."); time.sleep(espera)
                else:
                    resultados.append({"id": r["id"], "brazo": r["brazo"], "error": str(e)[:100]})
                    hechas.add((r["id"], r["brazo"])); guardar(); break
        print(f"[{i}/{len(lista)}] {r['id']}/{r['brazo']}")
        time.sleep(PAUSA)

lote = RESP[:N_PRUEBA] if MODO_PRUEBA else RESP
faltan = [r for r in lote if (r["id"], r["brazo"]) not in hechas]
print(f"{'PRUEBA' if MODO_PRUEBA else 'COMPLETO'}: {len(lote)} totales, {len(faltan)} pendientes, {len(hechas)} ya hechas")

await correr(lote)

# marcar completo
prev = json.load(open(OUT_PATH, encoding="utf-8"))
prev["meta"]["completo"] = (len([r for r in resultados if "error" not in r]) >= len(lote))
json.dump(prev, open(OUT_PATH,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\n✓ Guardado final: {OUT_PATH} ({len(resultados)} filas)")


COMPLETO: 171 totales, 171 pendientes, 0 ya hechas
[1/171] P1-01/A_baseline
[2/171] P1-01/B_graphrag
[3/171] P1-01/C_agente
[4/171] P1-02/A_baseline
[5/171] P1-02/B_graphrag
[6/171] P1-02/C_agente
[7/171] P1-03/A_baseline
[8/171] P1-03/B_graphrag
[9/171] P1-03/C_agente
[10/171] P1-04/A_baseline
[11/171] P1-04/B_graphrag
[12/171] P1-04/C_agente
[13/171] P1-05/A_baseline
[14/171] P1-05/B_graphrag
[15/171] P1-05/C_agente
[16/171] P1-06/A_baseline
[17/171] P1-06/B_graphrag
[18/171] P1-06/C_agente
[19/171] P1-07/A_baseline
[20/171] P1-07/B_graphrag
[21/171] P1-07/C_agente
[22/171] P1-08/A_baseline
[23/171] P1-08/B_graphrag
[24/171] P1-08/C_agente
[25/171] P1-09/A_baseline
[26/171] P1-09/B_graphrag
[27/171] P1-09/C_agente
[28/171] P1-10/A_baseline
[29/171] P1-10/B_graphrag
[30/171] P1-10/C_agente
[31/171] P1-11/A_baseline
[32/171] P1-11/B_graphrag
[33/171] P1-11/C_agente
[34/171] P1-12/A_baseline
[35/171] P1-12/B_graphrag
[36/171] P1-12/C_agente
[37/171] P1-13/A_baseline
[38/171] P1-13/B_gra

ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-EQXSAPAhN2ERQkJMiC7jQS60 on tokens per min (TPM): Limit 30000, Used 27839, Requested 2527. Please try again in 732ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-EQXSAPAhN2ERQkJMiC7jQS60 on tokens per min (TPM): Limit 30000, Used 27839, Requested 2527. Please try again in 732ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


[124/171] P2-21/A_baseline
[125/171] P2-21/B_graphrag
[126/171] P2-21/C_agente
[127/171] P2-22/A_baseline
[128/171] P2-22/B_graphrag
[129/171] P2-22/C_agente
[130/171] P2-23/A_baseline
[131/171] P2-23/B_graphrag
[132/171] P2-23/C_agente
[133/171] P2-24/A_baseline
[134/171] P2-24/B_graphrag
[135/171] P2-24/C_agente
[136/171] P2-25/A_baseline
[137/171] P2-25/B_graphrag
[138/171] P2-25/C_agente
[139/171] P2-26/A_baseline
[140/171] P2-26/B_graphrag
[141/171] P2-26/C_agente
[142/171] P2-27/A_baseline
[143/171] P2-27/B_graphrag
[144/171] P2-27/C_agente
[145/171] P2-28/A_baseline
[146/171] P2-28/B_graphrag
[147/171] P2-28/C_agente
[148/171] P2-29/A_baseline
[149/171] P2-29/B_graphrag
[150/171] P2-29/C_agente
[151/171] CT-01/A_baseline
[152/171] CT-01/B_graphrag
[153/171] CT-01/C_agente
[154/171] CT-02/A_baseline
[155/171] CT-02/B_graphrag
[156/171] CT-02/C_agente
[157/171] CT-03/A_baseline
[158/171] CT-03/B_graphrag


ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.o

[159/171] CT-03/C_agente


...

[169/171] CT-07/A_baseline


ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


[170/171] CT-07/B_graphrag


ERROR:instructor.v2.retry:API call failed on attempt 1: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


[171/171] CT-07/C_agente

✓ Guardado final: /content/drive/MyDrive/tesis_chatbot/ragas/metricas_ragas.json (171 filas)


## 6. Análisis — la tabla de la tesis

Promedios por brazo + el titular: detección de vigencia en las 17 preguntas de la reforma.

In [ ]:
# ============================================================================
# CELDA 6: ANÁLISIS — la tabla que va a la tesis
# ============================================================================
# Promedios por brazo de cada métrica, más el desglose de deteccion_de_vigencia
# sobre las preguntas afectadas por la reforma (el titular de la tesis).
import pandas as pd

df = pd.DataFrame(resultados)
if "error" in df.columns:
    err = df[df["error"].notna()]
    if len(err): print("⚠ Filas con error:", len(err)); display(err[["id","brazo","error"]])

MET_NUM = ["faithfulness","answer_relevancy","context_recall","context_precision"]

# --- 1) Promedios por brazo (métricas numéricas) ---
print("="*60); print("PROMEDIOS POR BRAZO (métricas RAGAS estándar)"); print("="*60)
tabla = df.groupby("brazo")[MET_NUM].mean(numeric_only=True).round(3)
display(tabla)

# --- 2) deteccion_de_vigencia: conteo por brazo ---
print("\n"+"="*60); print("DETECCIÓN DE VIGENCIA — conteo por brazo"); print("="*60)
piv = df.pivot_table(index="brazo", columns="deteccion_vigencia",
                     values="id", aggfunc="count", fill_value=0)
display(piv)

# --- 3) EL TITULAR: vigencia SOLO en las afectadas por la reforma ---
print("\n"+"="*60); print("★ TITULAR: vigencia en las preguntas de la REFORMA"); print("="*60)
ref = df[df["afectada_por_reforma"] == True]
if len(ref):
    piv_ref = ref.pivot_table(index="brazo", columns="deteccion_vigencia",
                              values="id", aggfunc="count", fill_value=0)
    display(piv_ref)
    # tasa de acierto por brazo
    print("\nTasa de 'correcto' sobre afectadas por reforma:")
    for b in sorted(ref["brazo"].unique()):
        sub = ref[ref["brazo"] == b]
        ok = (sub["deteccion_vigencia"] == "correcto").sum()
        print(f"  {b}: {ok}/{len(sub)}")

# --- 4) fallback en fuera_alcance ---
if "fallback_ok" in df.columns:
    print("\n"+"="*60); print("FALLBACK HONESTO (fuera_alcance)"); print("="*60)
    fa = df[df["fallback_ok"].notna()]
    for b in sorted(fa["brazo"].unique()):
        sub = fa[fa["brazo"] == b]
        ok = sub["fallback_ok"].sum()
        print(f"  {b}: {ok}/{len(sub)} admitió no saber")

# --- 5) guardar la tabla resumen ---
tabla.to_csv(os.path.join(CARPETA, "resumen_metricas.csv"))
print("\n✓ Resumen guardado en resumen_metricas.csv")


## Notas

- El juez GPT-4o evalúa respuestas de GPT-4o: sesgo de auto-preferencia SIMÉTRICO entre brazos (se cancela en la comparación). Documentar en Limitaciones.
- `deteccion_de_vigencia` sobre las afectadas por reforma = el resultado central.
- Salidas en Drive: `metricas_ragas.json`, `resumen_metricas.csv`.